In [4]:
df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [6]:
import kagglehub
import os 
# Download latest version
smap_path = kagglehub.dataset_download("team-ai/spam-text-message-classification")

print("Path to dataset files:", smap_path)

file = smap_path + '/' + os.listdir(smap_path)[0] 


import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

# ---------------------------------------------------------
# 1. Load and Prepare Data
# ---------------------------------------------------------
# Note: Adjust 'text' and 'label' to match the exact column names in the Kaggle CSV.
df = pd.read_csv(file)

# Handle any missing values to prevent training errors
df = df.dropna(subset=['Message', 'Category'])

X = df['Message']
y = df['Category'] # Assuming standard binary labels (e.g., 0 for Human, 1 for AI)

# 80/20 train-test split with stratification to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ---------------------------------------------------------
# 2. TF-IDF Tokenization
# ---------------------------------------------------------
print("Vectorizing text...")
tfidf = TfidfVectorizer(
    max_features=25000, 
    stop_words='english', 
    ngram_range=(1, 2) # Captures individual words and two-word phrases
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# ---------------------------------------------------------
# 3. Model Definition & Fine-Tuning
# ---------------------------------------------------------
print("Initializing base models...")
lr = LogisticRegression(max_iter=1000, random_state=42)
nb = MultinomialNB()
cb = CatBoostClassifier(iterations=300, verbose=0, random_state=42)
rf = RandomForestClassifier(random_state=42)

print("Fine-tuning Random Forest...")
# Example of fine-tuning Random Forest using RandomizedSearchCV
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 30, 50],
    'min_samples_split': [2, 5, 10]
}

# n_iter=5 keeps the tuning fast for demonstration; increase for better results
rf_tuned = RandomizedSearchCV(rf, rf_params, n_iter=5, cv=3, random_state=42, n_jobs=-1)
rf_tuned.fit(X_train_tfidf, y_train)
best_rf = rf_tuned.best_estimator_

# Fit the remaining base models
print("Training base models...")
lr.fit(X_train_tfidf, y_train)
nb.fit(X_train_tfidf, y_train)
cb.fit(X_train_tfidf, y_train)

# ---------------------------------------------------------
# 4. Ensemble Methods
# ---------------------------------------------------------
print("Building and training ensembles...")
estimators_list = [
    ('lr', lr), 
    ('nb', nb), 
    ('rf', best_rf), 
    ('cb', cb)
]

# A. Hard Voting (Majority Rules)
voting_hard = VotingClassifier(estimators=estimators_list, voting='hard')
voting_hard.fit(X_train_tfidf, y_train)

# B. Soft Voting (Averages the predicted probabilities)
voting_soft = VotingClassifier(estimators=estimators_list, voting='soft')
voting_soft.fit(X_train_tfidf, y_train)

# C. Stacking (Uses a meta-model to learn how to best combine base model predictions)
stacking = StackingClassifier(
    estimators=estimators_list,
    final_estimator=LogisticRegression(),
    n_jobs=-1
)
stacking.fit(X_train_tfidf, y_train)

# ---------------------------------------------------------
# 5. Evaluation and Selection
# ---------------------------------------------------------
print("\n--- Model Performance ---")
models = {
    "Logistic Regression": lr,
    "Naive Bayes": nb,
    "Random Forest (Tuned)": best_rf,
    "CatBoost": cb,
    "Hard Voting Ensemble": voting_hard,
    "Soft Voting Ensemble": voting_soft,
    "Stacking Ensemble": stacking
}

for name, model in models.items():
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name:25} Accuracy: {acc:.4f}")

Path to dataset files: /Users/rafael/.cache/kagglehub/datasets/team-ai/spam-text-message-classification/versions/1
Vectorizing text...
Initializing base models...
Fine-tuning Random Forest...
Training base models...
Building and training ensembles...

--- Model Performance ---
Logistic Regression       Accuracy: 0.9623
Naive Bayes               Accuracy: 0.9614
Random Forest (Tuned)     Accuracy: 0.9632
CatBoost                  Accuracy: 0.9740
Hard Voting Ensemble      Accuracy: 0.9605
Soft Voting Ensemble      Accuracy: 0.9677
Stacking Ensemble         Accuracy: 0.9821


In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import random

# Force deterministic behavior for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# ---------------------------------------------------------
# 1. Load and Prepare Data
# ---------------------------------------------------------
print("Loading data...")
df = pd.read_csv(file)

# Handle missing values and ensure clean string types
df = df.dropna(subset=['Message', 'Category'])
df['Message'] = df['Message'].astype(str)

# Ensure Category is mapped to numeric integers (e.g., 0 and 1)
if df['Category'].dtype == 'object' or str(df['Category'].dtype) == 'category':
    df['Category'] = df['Category'].astype('category').cat.codes

# BUG FIX: Explicitly convert to NumPy arrays to avoid Pandas Series indexing bugs downstream
X = df['Message'].to_numpy()
y = df['Category'].to_numpy()

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ---------------------------------------------------------
# 2. BERT Training Component
# ---------------------------------------------------------
print("\n--- Starting BERT Fine-Tuning ---")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load Pre-trained Tokenizer and Model
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# PyTorch Dataset Wrapper
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __getitem__(self, idx):
        # BUG FIX: Force the index to be a standard Python integer.
        # This prevents the TypeError when PyTorch or NumPy passes a scalar tensor or int64 type.
        idx = int(idx.item()) if torch.is_tensor(idx) else int(idx)
        
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(X_train, y_train, tokenizer)
test_dataset = TextDataset(X_test, y_test, tokenizer)

# Define Training Configurations
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",       # <--- FIX: Updated parameter name
    save_strategy="epoch",       # <--- Matches eval_strategy
    load_best_model_at_end=True,
    report_to="none" 
)

# Initialize Hugging Face Trainer
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# Train the model
trainer.train()

# Extract Class Probabilities from BERT for Ensembling
print("Extracting BERT predictions...")
raw_predictions = trainer.predict(test_dataset)
# Apply softmax to get proper 0-1 probabilities for both classes
bert_probs = torch.nn.functional.softmax(torch.tensor(raw_predictions.predictions), dim=-1).numpy()
bert_preds = np.argmax(bert_probs, axis=1)

# ---------------------------------------------------------
# 3. TF-IDF Tokenization (for Classical Models)
# ---------------------------------------------------------
print("\n--- Processing Text with TF-IDF ---")
tfidf = TfidfVectorizer(max_features=25000, stop_words='english', ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# ---------------------------------------------------------
# 4. Classical Model Architecture & Tuning
# ---------------------------------------------------------
print("Training Classical Classifiers...")
lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_tfidf, y_train)
nb = MultinomialNB().fit(X_train_tfidf, y_train)
cb = CatBoostClassifier(iterations=300, verbose=0, random_state=42).fit(X_train_tfidf, y_train)

print("Fine-tuning Random Forest...")
rf = RandomForestClassifier(random_state=42)
rf_params = {'n_estimators': [100, 200], 'max_depth': [None, 20], 'min_samples_split': [2, 5]}
rf_tuned = RandomizedSearchCV(rf, rf_params, n_iter=4, cv=3, random_state=42, n_jobs=-1).fit(X_train_tfidf, y_train)
best_rf = rf_tuned.best_estimator_

# ---------------------------------------------------------
# 5. Standard Classical Ensembles
# ---------------------------------------------------------
print("Building ensembles...")
estimators_list = [('lr', lr), ('nb', nb), ('rf', best_rf), ('cb', cb)]

voting_hard = VotingClassifier(estimators=estimators_list, voting='hard').fit(X_train_tfidf, y_train)
voting_soft = VotingClassifier(estimators=estimators_list, voting='soft').fit(X_train_tfidf, y_train)
stacking = StackingClassifier(estimators=estimators_list, final_estimator=LogisticRegression(), n_jobs=-1).fit(X_train_tfidf, y_train)

# ---------------------------------------------------------
# 6. Comprehensive Performance Evaluation
# ---------------------------------------------------------
print("\n=== Final Performance Metrics ===")

# Evaluate standard models
models = {
    "Logistic Regression": lr.predict(X_test_tfidf),
    "Naive Bayes": nb.predict(X_test_tfidf),
    "Random Forest (Tuned)": best_rf.predict(X_test_tfidf),
    "CatBoost": cb.predict(X_test_tfidf),
    "Hard Voting Ensemble": voting_hard.predict(X_test_tfidf),
    "Soft Voting Ensemble": voting_soft.predict(X_test_tfidf),
    "Stacking Ensemble": stacking.predict(X_test_tfidf),
    "BERT Finetuned": bert_preds
}

for name, predictions in models.items():
    acc = accuracy_score(y_test, predictions)
    print(f"{name:25} Accuracy: {acc:.4f}")

# ---------------------------------------------------------
# 7. Advanced Meta-Ensemble (Classical Models + BERT)
# ---------------------------------------------------------
# Combine probabilities of the best systems to make a hybrid decision
print("\nEvaluating Hybrid Meta-Ensemble (Soft Voting + BERT)...")
soft_voting_probs = voting_soft.predict_proba(X_test_tfidf)

# Blend probabilities (Equal weight distribution example)
hybrid_probs = (soft_voting_probs + bert_probs) / 2
hybrid_preds = np.argmax(hybrid_probs, axis=1)
hybrid_acc = accuracy_score(y_test, hybrid_preds)

print(f"Hybrid Meta-Ensemble Accuracy: {hybrid_acc:.4f}")

Loading data...

--- Starting BERT Fine-Tuning ---
Using device: cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12548.17it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

ValueError: --load_best_model_at_end requires the save and eval strategy to match, except when --save_strategy="best", but found
- Evaluation strategy: IntervalStrategy.NO
- Save strategy: SaveStrategy.EPOCH